<a href="https://colab.research.google.com/github/scottaudrey/svd-food-recommendation-system/blob/main/notebooks/01_ratings_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
foods = ["Sweet 1", "Sweet 2", "Sweet 3",
          "Savory 1", "Savory 2", "Savory 3"]
users = ["John", "Ben", "Jake", "Princeton", "Eve", "Fay"]

In [9]:
ratings = np.array([
[5, 4, 5, 1, np.nan, 2], # John
[4, np.nan, 5, 2, 1, 1], # Ben
[5, 5, np.nan, 1, 2, np.nan], # Jake
[1, 2, 1, 5, 4, np.nan], # Princeton
[np.nan, 1, 2, 4, 5, 5], # Eve
[2, 1, np.nan, 5, np.nan, 4], # Fay
])

In [10]:
ratings_table = pd.DataFrame(ratings, index=users, columns=foods)
ratings_table

,Sweet 1,Sweet 2,Sweet 3,Savory 1,Savory 2,Savory 3
John,5.0,4.0,5.0,1.0,NaN,2.0
Ben,4.0,NaN,5.0,2.0,1.0,1.0
Jake,5.0,5.0,NaN,1.0,2.0,NaN
Princeton,1.0,2.0,1.0,5.0,4.0,NaN
Eve,NaN,1.0,2.0,4.0,5.0,5.0
Fay,2.0,1.0,NaN,5.0,NaN,4.0


In [11]:
missing = np.isnan(ratings).sum()
print("Missing entries:", missing, "out of", ratings.size)

Missing entries: 8 out of 36


## Part B: The Amazon Food Review Data

In [13]:
import os
import urlib.request
import zipfile

if not os.path.exists("reviews.csv"):
  urlib.request.urlretrieve(
      https://
  )

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

## Part C: Building the user-by-food matrix

In [12]:
assert isinstance(ratings, pd.DataFrame), "Run Part B first: `ratings` must be the food ratings DataFrame"
required = {"user", "food", "rating"}
missing_cols = required - set(ratings.columns)
assert not missing_cols, f"`ratings` is missing columns: {missing_cols}"

AssertionError: Run Part B first: `ratings` must be the food ratings DataFrame

In [ ]:
n_users = ratings["user"].nunique()
n_foods = ratings["food"].nunique()
print("Matrix would be", n_users, "x", n_foods, "=", n_users * n_foods, "cells")

MAX_CELLS = 20_000_000
min_ratings = 20

if n_users * n_foods > MAX_CELLS:
    food_counts = ratings["food"].value_counts()
    user_counts = ratings["user"].value_counts()
    ratings = ratings[
        ratings["food"].isin(food_counts[food_counts >= min_ratings].index) &
        ratings["user"].isin(user_counts[user_counts >= min_ratings].index)
    ]
    n_users = ratings["user"].nunique()
    n_foods = ratings["food"].nunique()
    print("After filtering:", n_users, "x", n_foods, "=", n_users * n_foods, "cells")

In [ ]:
R = ratings.pivot_table(
    index="user",
    columns="food",
    values="rating"
)

In [ ]:
print("Matrix shape:", R.shape)
fraction_missing = R.isna().to_numpy().mean()
print("Fraction missing:", round(fraction_missing, 4))

In [ ]:
per_user = ratings.groupby("user").size()
print("Least active user:", per_user.min())
print("Median user:", int(per_user.median()))
print("Most active user:", per_user.max())

In [ ]:
print("Every rating in R:", R.notna().sum().sum() == len(ratings))

## Part D: Which Foods are Rated Most

In [ ]:
foods = pd.read_csv(
    "m1-100k/u.food",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["food", "title"]
)

In [ ]:
stats = ratings.groupby("food")["rating"].agg(["count", "mean"])
stats = stats.join(foods.set_index("food"))

top_five = stats.sort_values("count", ascending=False).head(5)
top_five.round(2)

## Exercises

In [ ]:
5.